In [3]:
# ============================================================
# 1. 라이브러리
# ============================================================

import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


# ============================================================
# 2. 설정값
# ============================================================

SEED = 42
N_SPLITS = 5
BATCH_SIZE = 32
MAX_EPOCHS = 200
PATIENCE = 20

# FT-Transformer 설정
D_TOKEN = 64
N_HEADS = 8
N_LAYERS = 2
DROPOUT = 0.1

LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.0001


# ============================================================
# 3. 랜덤 시드 고정
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# 4. 실행 장치 설정
# ============================================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("사용 장치:", device)


# ============================================================
# 5. 데이터 불러오기
# ============================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train 크기:", train.shape)
print("Test 크기:", test.shape)


# ============================================================
# 6. X, y 분리
# ============================================================

y = train["SalePrice"]

x = train.drop(
    columns=["Id", "SalePrice"]
)

test_id = test["Id"]

x_test = test.drop(
    columns=["Id"]
)

print("X 크기:", x.shape)
print("y 크기:", y.shape)
print("Test X 크기:", x_test.shape)


# ============================================================
# 7. Feature Engineering
# ============================================================

def add_features(data):

    data = data.copy()

    # 집이 판매될 당시의 집 나이
    data["HouseAge"] = (
        data["YrSold"]
        - data["YearBuilt"]
    )

    # 지하실 욕실을 포함한 전체 욕실 수
    data["TotalBath"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    return data


x = add_features(x)
x_test = add_features(x_test)


# ============================================================
# 8. 타깃 로그 변환
# ============================================================

y_log = np.log1p(y)


# ============================================================
# 9. Feature Tokenizer
# ============================================================

class FeatureTokenizer(nn.Module):

    def __init__(
        self,
        n_numeric_features,
        category_cardinalities,
        d_token
    ):
        super().__init__()

        self.n_numeric_features = (
            n_numeric_features
        )

        self.n_categorical_features = len(
            category_cardinalities
        )

        self.d_token = d_token

        # ----------------------------------------------------
        # 수치형 피처용 학습 가능한 가중치
        # ----------------------------------------------------

        if n_numeric_features > 0:

            self.numeric_weight = nn.Parameter(
                torch.empty(
                    n_numeric_features,
                    d_token
                )
            )

            self.numeric_bias = nn.Parameter(
                torch.empty(
                    n_numeric_features,
                    d_token
                )
            )

        else:

            self.numeric_weight = None
            self.numeric_bias = None


        # ----------------------------------------------------
        # 범주형 피처용 Embedding
        # ----------------------------------------------------

        if self.n_categorical_features > 0:

            total_categories = sum(
                category_cardinalities
            )

            self.category_embedding = nn.Embedding(
                total_categories,
                d_token
            )

            # 각 범주형 컬럼의 번호가 겹치지 않도록
            # 시작 번호를 계산
            category_offsets = np.array(
                [0]
                + list(
                    np.cumsum(
                        category_cardinalities[:-1]
                    )
                ),
                dtype=np.int64
            )

            self.register_buffer(
                "category_offsets",
                torch.tensor(
                    category_offsets,
                    dtype=torch.long
                )
            )

        else:

            self.category_embedding = None


        # ----------------------------------------------------
        # 최종 예측을 담당할 CLS 토큰
        # ----------------------------------------------------

        self.cls_token = nn.Parameter(
            torch.empty(
                1,
                1,
                d_token
            )
        )

        self.reset_parameters()


    def reset_parameters(self):

        if self.numeric_weight is not None:

            nn.init.normal_(
                self.numeric_weight,
                mean=0.0,
                std=0.02
            )

            nn.init.normal_(
                self.numeric_bias,
                mean=0.0,
                std=0.02
            )

        if self.category_embedding is not None:

            nn.init.normal_(
                self.category_embedding.weight,
                mean=0.0,
                std=0.02
            )

        nn.init.normal_(
            self.cls_token,
            mean=0.0,
            std=0.02
        )


    def forward(
        self,
        x_numeric,
        x_categorical
    ):

        tokens = []

        batch_size = x_numeric.shape[0]


        # ----------------------------------------------------
        # 수치형 값을 토큰으로 변환
        # ----------------------------------------------------

        if self.n_numeric_features > 0:

            # x_numeric:
            # (batch, numeric_features)
            #
            # numeric_tokens:
            # (batch, numeric_features, d_token)

            numeric_tokens = (
                x_numeric.unsqueeze(-1)
                * self.numeric_weight.unsqueeze(0)
                + self.numeric_bias.unsqueeze(0)
            )

            tokens.append(
                numeric_tokens
            )


        # ----------------------------------------------------
        # 범주형 값을 Embedding 토큰으로 변환
        # ----------------------------------------------------

        if self.n_categorical_features > 0:

            categorical_indices = (
                x_categorical
                + self.category_offsets.unsqueeze(0)
            )

            categorical_tokens = (
                self.category_embedding(
                    categorical_indices
                )
            )

            tokens.append(
                categorical_tokens
            )


        # 수치형 토큰과 범주형 토큰 연결
        feature_tokens = torch.cat(
            tokens,
            dim=1
        )


        # CLS 토큰을 각 집 데이터 앞에 추가
        cls_tokens = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        all_tokens = torch.cat(
            [
                cls_tokens,
                feature_tokens
            ],
            dim=1
        )

        return all_tokens


# ============================================================
# 10. FT-Transformer 모델
# ============================================================

class FTTransformer(nn.Module):

    def __init__(
        self,
        n_numeric_features,
        category_cardinalities,
        d_token=64,
        n_heads=8,
        n_layers=2,
        dropout=0.1
    ):
        super().__init__()

        # 각 피처를 토큰으로 변경
        self.tokenizer = FeatureTokenizer(
            n_numeric_features=(
                n_numeric_features
            ),
            category_cardinalities=(
                category_cardinalities
            ),
            d_token=d_token
        )


        # Transformer Encoder 한 층의 구조
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )


        # Transformer Encoder 2층
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layers
        )


        # Transformer 결과 정규화
        self.final_norm = nn.LayerNorm(
            d_token
        )


        # 회귀 Head
        #
        # hidden layer를 두 개 사용:
        # d_token → 64 → 32 → 1
        self.regression_head = nn.Sequential(
            nn.Linear(
                d_token,
                64
            ),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(
                64,
                32
            ),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                1
            )
        )


    def forward(
        self,
        x_numeric,
        x_categorical
    ):

        # 피처를 토큰으로 변환
        tokens = self.tokenizer(
            x_numeric,
            x_categorical
        )

        # 피처 사이의 관계 학습
        transformed_tokens = self.transformer(
            tokens
        )

        # 첫 번째 위치인 CLS 토큰만 사용
        cls_output = transformed_tokens[:, 0]

        cls_output = self.final_norm(
            cls_output
        )

        prediction = self.regression_head(
            cls_output
        )

        return prediction.squeeze(1)


# ============================================================
# 11. 5-Fold 준비
# ============================================================

kfold = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

# OOF 예측값
oof_predictions = np.zeros(
    len(x),
    dtype=np.float64
)

# 각 Fold의 test 예측값
test_fold_predictions = []

fold_scores = []


# ============================================================
# 12. 5-Fold 시작
# ============================================================

for fold, (train_idx, valid_idx) in enumerate(
    kfold.split(x),
    start=1
):

    print()
    print("=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)


    # --------------------------------------------------------
    # 12-1. Fold 데이터 분리
    # --------------------------------------------------------

    x_train_fold = (
        x.iloc[train_idx]
        .copy()
    )

    x_valid_fold = (
        x.iloc[valid_idx]
        .copy()
    )

    x_test_fold = x_test.copy()

    y_train_fold = (
        y_log.iloc[train_idx]
        .copy()
    )

    y_valid_fold = (
        y_log.iloc[valid_idx]
        .copy()
    )


    # --------------------------------------------------------
    # 12-2. 타깃 중심화
    # --------------------------------------------------------

    target_mean = float(
        y_train_fold.mean()
    )

    y_train_centered = (
        y_train_fold
        - target_mean
    )

    y_valid_centered = (
        y_valid_fold
        - target_mean
    )

    print("Target 평균:", target_mean)
    print(
        "중심화 후 평균:",
        y_train_centered.mean()
    )


    # --------------------------------------------------------
    # 12-3. 수치형과 범주형 구분
    # --------------------------------------------------------

    numeric_columns = (
        x_train_fold
        .select_dtypes(
            include=["number"]
        )
        .columns
        .tolist()
    )

    categorical_columns = (
        x_train_fold
        .select_dtypes(
            exclude=["number"]
        )
        .columns
        .tolist()
    )

    print(
        "수치형 피처:",
        len(numeric_columns)
    )

    print(
        "범주형 피처:",
        len(categorical_columns)
    )


    # --------------------------------------------------------
    # 12-4. 수치형 결측값 처리
    # --------------------------------------------------------

    # 현재 Fold의 train 중앙값만 사용
    numeric_medians = (
        x_train_fold[numeric_columns]
        .median()
    )

    for data in [
        x_train_fold,
        x_valid_fold,
        x_test_fold
    ]:

        data[numeric_columns] = (
            data[numeric_columns]
            .fillna(numeric_medians)
        )

        data[categorical_columns] = (
            data[categorical_columns]
            .fillna("Missing")
            .astype(str)
        )


    # --------------------------------------------------------
    # 12-5. 수치형 StandardScaler
    # --------------------------------------------------------

    numeric_scaler = StandardScaler()

    x_train_numeric = (
        numeric_scaler.fit_transform(
            x_train_fold[
                numeric_columns
            ]
        )
    )

    x_valid_numeric = (
        numeric_scaler.transform(
            x_valid_fold[
                numeric_columns
            ]
        )
    )

    x_test_numeric = (
        numeric_scaler.transform(
            x_test_fold[
                numeric_columns
            ]
        )
    )


    # --------------------------------------------------------
    # 12-6. 범주형 Ordinal Encoding
    # --------------------------------------------------------

    # One-Hot Encoding을 사용하지 않고
    # 각 범주에 번호를 부여함
    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    x_train_categorical = (
        categorical_encoder.fit_transform(
            x_train_fold[
                categorical_columns
            ]
        )
    )

    x_valid_categorical = (
        categorical_encoder.transform(
            x_valid_fold[
                categorical_columns
            ]
        )
    )

    x_test_categorical = (
        categorical_encoder.transform(
            x_test_fold[
                categorical_columns
            ]
        )
    )


    # OrdinalEncoder:
    # 알려진 범주 = 0, 1, 2 ...
    # 처음 보는 범주 = -1
    #
    # 여기에 1을 더해서:
    # 처음 보는 범주 = 0
    # 알려진 범주 = 1, 2, 3 ...

    x_train_categorical = (
        x_train_categorical.astype(np.int64)
        + 1
    )

    x_valid_categorical = (
        x_valid_categorical.astype(np.int64)
        + 1
    )

    x_test_categorical = (
        x_test_categorical.astype(np.int64)
        + 1
    )


    # 각 범주형 피처가 가질 수 있는
    # 전체 범주 개수
    #
    # +1은 Unknown 범주를 위한 공간
    category_cardinalities = [
        len(categories) + 1
        for categories
        in categorical_encoder.categories_
    ]

    print(
        "전체 범주 Embedding 개수:",
        sum(category_cardinalities)
    )


    # --------------------------------------------------------
    # 12-7. Tensor 변환
    # --------------------------------------------------------

    x_train_numeric_tensor = torch.tensor(
        x_train_numeric,
        dtype=torch.float32
    )

    x_valid_numeric_tensor = torch.tensor(
        x_valid_numeric,
        dtype=torch.float32
    )

    x_test_numeric_tensor = torch.tensor(
        x_test_numeric,
        dtype=torch.float32
    )

    x_train_categorical_tensor = torch.tensor(
        x_train_categorical,
        dtype=torch.long
    )

    x_valid_categorical_tensor = torch.tensor(
        x_valid_categorical,
        dtype=torch.long
    )

    x_test_categorical_tensor = torch.tensor(
        x_test_categorical,
        dtype=torch.long
    )

    y_train_tensor = torch.tensor(
        y_train_centered.to_numpy(),
        dtype=torch.float32
    )

    y_valid_tensor = torch.tensor(
        y_valid_centered.to_numpy(),
        dtype=torch.float32
    )


    # --------------------------------------------------------
    # 12-8. Dataset과 DataLoader
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        x_train_numeric_tensor,
        x_train_categorical_tensor,
        y_train_tensor
    )

    valid_dataset = TensorDataset(
        x_valid_numeric_tensor,
        x_valid_categorical_tensor,
        y_valid_tensor
    )

    generator = torch.Generator()
    generator.manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )


    # --------------------------------------------------------
    # 12-9. Fold마다 새로운 FT-Transformer 생성
    # --------------------------------------------------------

    torch.manual_seed(SEED + fold)

    model = FTTransformer(
        n_numeric_features=len(
            numeric_columns
        ),
        category_cardinalities=(
            category_cardinalities
        ),
        d_token=D_TOKEN,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT
    ).to(device)


    # --------------------------------------------------------
    # 12-10. Loss와 Optimizer
    # --------------------------------------------------------

    # 첫 실험에서는 기존 MLP와 동일하게 MSE 사용
    criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    # --------------------------------------------------------
    # 12-11. Early Stopping 설정
    # --------------------------------------------------------

    best_valid_rmse = float("inf")

    patience_counter = 0

    checkpoint_path = (
        f"best_ft_transformer_fold_{fold}.pt"
    )


    # ========================================================
    # 13. 학습 Loop
    # ========================================================

    for epoch in range(MAX_EPOCHS):

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model.train()

        total_train_squared_error = 0

        for (
            batch_numeric,
            batch_categorical,
            batch_y
        ) in train_loader:

            batch_numeric = (
                batch_numeric.to(device)
            )

            batch_categorical = (
                batch_categorical.to(device)
            )

            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            predictions = model(
                batch_numeric,
                batch_categorical
            )

            loss = criterion(
                predictions,
                batch_y
            )

            loss.backward()

            # Transformer에서 갑자기 너무 큰
            # gradient가 생기는 것을 방지
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_train_squared_error += (
                (predictions.detach() - batch_y) ** 2
            ).sum().item()


        train_rmse = np.sqrt(
            total_train_squared_error
            / len(train_loader.dataset)
        )


        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        total_valid_squared_error = 0

        with torch.no_grad():

            for (
                batch_numeric,
                batch_categorical,
                batch_y
            ) in valid_loader:

                batch_numeric = (
                    batch_numeric.to(device)
                )

                batch_categorical = (
                    batch_categorical.to(device)
                )

                batch_y = batch_y.to(device)

                predictions = model(
                    batch_numeric,
                    batch_categorical
                )

                total_valid_squared_error += (
                    (predictions - batch_y) ** 2
                ).sum().item()


        valid_rmse = np.sqrt(
            total_valid_squared_error
            / len(valid_loader.dataset)
        )


        # ----------------------------------------------------
        # 학습 결과 출력
        # ----------------------------------------------------

        print(
            f"Fold {fold} | "
            f"Epoch {epoch + 1:3d} | "
            f"Train RMSE={train_rmse:.5f} | "
            f"Valid RMSE={valid_rmse:.5f}"
        )


        # ----------------------------------------------------
        # Best Checkpoint 저장
        # ----------------------------------------------------

        if valid_rmse < best_valid_rmse:

            best_valid_rmse = valid_rmse
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print("  -> Best Model 저장")

        else:

            patience_counter += 1


        # ----------------------------------------------------
        # Early Stopping
        # ----------------------------------------------------

        if patience_counter >= PATIENCE:

            print(
                f"Fold {fold} Early Stopping!"
            )

            break


    # ========================================================
    # 14. 최고 모델 불러오기
    # ========================================================

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )
    )

    model.eval()


    # ========================================================
    # 15. Validation과 Test 예측
    # ========================================================

    with torch.no_grad():

        valid_centered_predictions = (
            model(
                x_valid_numeric_tensor.to(device),
                x_valid_categorical_tensor.to(device)
            )
            .cpu()
            .numpy()
        )

        test_centered_predictions = (
            model(
                x_test_numeric_tensor.to(device),
                x_test_categorical_tensor.to(device)
            )
            .cpu()
            .numpy()
        )


    # 중심화할 때 뺀 target 평균 복구
    valid_log_predictions = (
        valid_centered_predictions
        + target_mean
    )

    fold_test_log_predictions = (
        test_centered_predictions
        + target_mean
    )


    # OOF 예측 저장
    oof_predictions[valid_idx] = (
        valid_log_predictions
    )

    # Test Fold 예측 저장
    test_fold_predictions.append(
        fold_test_log_predictions
    )


    # --------------------------------------------------------
    # 15-1. Fold RMSE
    # --------------------------------------------------------

    fold_rmse = np.sqrt(
        np.mean(
            (
                valid_log_predictions
                - y_valid_fold.to_numpy()
            ) ** 2
        )
    )

    fold_scores.append(
        fold_rmse
    )

    print()
    print(
        f"Fold {fold} Best RMSE: "
        f"{fold_rmse:.5f}"
    )


# ============================================================
# 16. 전체 OOF RMSE
# ============================================================

oof_rmse = np.sqrt(
    np.mean(
        (
            oof_predictions
            - y_log.to_numpy()
        ) ** 2
    )
)

fold_mean_rmse = np.mean(
    fold_scores
)

fold_std_rmse = np.std(
    fold_scores
)

print()
print("=" * 60)
print("FT-Transformer 최종 결과")
print("=" * 60)

for fold, score in enumerate(
    fold_scores,
    start=1
):

    print(
        f"Fold {fold}: {score:.5f}"
    )

print(
    f"Fold 평균 RMSE: {fold_mean_rmse:.5f}"
)

print(
    f"Fold 표준편차: {fold_std_rmse:.5f}"
)

print(
    f"전체 OOF RMSE: {oof_rmse:.5f}"
)


# ============================================================
# 17. 5개 Fold Test 예측 평균
# ============================================================

test_log_predictions = np.mean(
    np.stack(
        test_fold_predictions,
        axis=0
    ),
    axis=0
)


# ============================================================
# 18. 실제 가격으로 복구
# ============================================================

test_predictions = np.expm1(
    test_log_predictions
)

test_predictions = np.maximum(
    test_predictions,
    0
)


# ============================================================
# 19. 제출 파일 생성
# ============================================================

submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

print()
print(
    "submission.csv 생성 완료!"
)

사용 장치: mps
Train 크기: (1460, 81)
Test 크기: (1459, 80)
X 크기: (1460, 79)
y 크기: (1460,)
Test X 크기: (1459, 79)

Fold 1
Target 평균: 12.030658310971573
중심화 후 평균: 4.258389683493751e-16
수치형 피처: 38
범주형 피처: 43
전체 범주 Embedding 개수: 308


/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_32074/2830362970.py:386: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Fold 1 | Epoch   1 | Train RMSE=0.28206 | Valid RMSE=0.20859
  -> Best Model 저장
Fold 1 | Epoch   2 | Train RMSE=0.19484 | Valid RMSE=0.18288
  -> Best Model 저장
Fold 1 | Epoch   3 | Train RMSE=0.16186 | Valid RMSE=0.16127
  -> Best Model 저장
Fold 1 | Epoch   4 | Train RMSE=0.14929 | Valid RMSE=0.14808
  -> Best Model 저장
Fold 1 | Epoch   5 | Train RMSE=0.14137 | Valid RMSE=0.15191
Fold 1 | Epoch   6 | Train RMSE=0.13977 | Valid RMSE=0.14010
  -> Best Model 저장
Fold 1 | Epoch   7 | Train RMSE=0.13200 | Valid RMSE=0.15148
Fold 1 | Epoch   8 | Train RMSE=0.13281 | Valid RMSE=0.14106
Fold 1 | Epoch   9 | Train RMSE=0.12289 | Valid RMSE=0.16629
Fold 1 | Epoch  10 | Train RMSE=0.12741 | Valid RMSE=0.16431
Fold 1 | Epoch  11 | Train RMSE=0.12513 | Valid RMSE=0.15173
Fold 1 | Epoch  12 | Train RMSE=0.12174 | Valid RMSE=0.14552
Fold 1 | Epoch  13 | Train RMSE=0.11835 | Valid RMSE=0.15108
Fold 1 | Epoch  14 | Train RMSE=0.11765 | Valid RMSE=0.14858
Fold 1 | Epoch  15 | Train RMSE=0.11656 | Valid RMS

/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_32074/2830362970.py:386: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Fold 2 | Epoch   1 | Train RMSE=0.29874 | Valid RMSE=0.20414
  -> Best Model 저장
Fold 2 | Epoch   2 | Train RMSE=0.19270 | Valid RMSE=0.14354
  -> Best Model 저장
Fold 2 | Epoch   3 | Train RMSE=0.17169 | Valid RMSE=0.13602
  -> Best Model 저장
Fold 2 | Epoch   4 | Train RMSE=0.15680 | Valid RMSE=0.13165
  -> Best Model 저장
Fold 2 | Epoch   5 | Train RMSE=0.14464 | Valid RMSE=0.13820
Fold 2 | Epoch   6 | Train RMSE=0.14449 | Valid RMSE=0.13313
Fold 2 | Epoch   7 | Train RMSE=0.14032 | Valid RMSE=0.12270
  -> Best Model 저장
Fold 2 | Epoch   8 | Train RMSE=0.13395 | Valid RMSE=0.13074
Fold 2 | Epoch   9 | Train RMSE=0.14014 | Valid RMSE=0.12452
Fold 2 | Epoch  10 | Train RMSE=0.13343 | Valid RMSE=0.12020
  -> Best Model 저장
Fold 2 | Epoch  11 | Train RMSE=0.13425 | Valid RMSE=0.12639
Fold 2 | Epoch  12 | Train RMSE=0.13084 | Valid RMSE=0.12581
Fold 2 | Epoch  13 | Train RMSE=0.12286 | Valid RMSE=0.12170
Fold 2 | Epoch  14 | Train RMSE=0.12045 | Valid RMSE=0.14004
Fold 2 | Epoch  15 | Train RMSE=

...

Fold 3 | Epoch   1 | Train RMSE=0.27251 | Valid RMSE=0.17557
  -> Best Model 저장
Fold 3 | Epoch   2 | Train RMSE=0.18329 | Valid RMSE=0.14843
  -> Best Model 저장
Fold 3 | Epoch   3 | Train RMSE=0.15887 | Valid RMSE=0.15414
Fold 3 | Epoch   4 | Train RMSE=0.14651 | Valid RMSE=0.15042
Fold 3 | Epoch   5 | Train RMSE=0.14145 | Valid RMSE=0.16699
Fold 3 | Epoch   6 | Train RMSE=0.14404 | Valid RMSE=0.15304
Fold 3 | Epoch   7 | Train RMSE=0.12897 | Valid RMSE=0.14971
Fold 3 | Epoch   8 | Train RMSE=0.12687 | Valid RMSE=0.14499
  -> Best Model 저장
Fold 3 | Epoch   9 | Train RMSE=0.12868 | Valid RMSE=0.14837
Fold 3 | Epoch  10 | Train RMSE=0.12898 | Valid RMSE=0.15195
Fold 3 | Epoch  11 | Train RMSE=0.12136 | Valid RMSE=0.15285
Fold 3 | Epoch  12 | Train RMSE=0.12350 | Valid RMSE=0.15338
Fold 3 | Epoch  13 | Train RMSE=0.12019 | Valid RMSE=0.16609
Fold 3 | Epoch  14 | Train RMSE=0.11997 | Valid RMSE=0.15166
Fold 3 | Epoch  15 | Train RMSE=0.12031 | Valid RMSE=0.15115
Fold 3 | Epoch  16 | Train R

/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_32074/2830362970.py:386: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Fold 4 | Epoch   1 | Train RMSE=0.28416 | Valid RMSE=0.18922
  -> Best Model 저장
Fold 4 | Epoch   2 | Train RMSE=0.19429 | Valid RMSE=0.15187
  -> Best Model 저장
Fold 4 | Epoch   3 | Train RMSE=0.16852 | Valid RMSE=0.15882
Fold 4 | Epoch   4 | Train RMSE=0.15783 | Valid RMSE=0.14074
  -> Best Model 저장
Fold 4 | Epoch   5 | Train RMSE=0.14167 | Valid RMSE=0.15297
Fold 4 | Epoch   6 | Train RMSE=0.14737 | Valid RMSE=0.14210
Fold 4 | Epoch   7 | Train RMSE=0.14058 | Valid RMSE=0.13553
  -> Best Model 저장
Fold 4 | Epoch   8 | Train RMSE=0.13266 | Valid RMSE=0.14510
Fold 4 | Epoch   9 | Train RMSE=0.13893 | Valid RMSE=0.12964
  -> Best Model 저장
Fold 4 | Epoch  10 | Train RMSE=0.12892 | Valid RMSE=0.13191
Fold 4 | Epoch  11 | Train RMSE=0.12604 | Valid RMSE=0.12988
Fold 4 | Epoch  12 | Train RMSE=0.12954 | Valid RMSE=0.12955
  -> Best Model 저장
Fold 4 | Epoch  13 | Train RMSE=0.12790 | Valid RMSE=0.12842
  -> Best Model 저장
Fold 4 | Epoch  14 | Train RMSE=0.12292 | Valid RMSE=0.13281
Fold 4 | Epoc

/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_32074/2830362970.py:386: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Fold 5 | Epoch   1 | Train RMSE=0.26780 | Valid RMSE=0.15392
  -> Best Model 저장
Fold 5 | Epoch   2 | Train RMSE=0.19348 | Valid RMSE=0.12549
  -> Best Model 저장
Fold 5 | Epoch   3 | Train RMSE=0.16799 | Valid RMSE=0.12663
Fold 5 | Epoch   4 | Train RMSE=0.15295 | Valid RMSE=0.12825
Fold 5 | Epoch   5 | Train RMSE=0.13892 | Valid RMSE=0.11569
  -> Best Model 저장
Fold 5 | Epoch   6 | Train RMSE=0.14539 | Valid RMSE=0.12826
Fold 5 | Epoch   7 | Train RMSE=0.13580 | Valid RMSE=0.11393
  -> Best Model 저장
Fold 5 | Epoch   8 | Train RMSE=0.13322 | Valid RMSE=0.11640
Fold 5 | Epoch   9 | Train RMSE=0.12888 | Valid RMSE=0.11217
  -> Best Model 저장
Fold 5 | Epoch  10 | Train RMSE=0.12870 | Valid RMSE=0.12213
Fold 5 | Epoch  11 | Train RMSE=0.12434 | Valid RMSE=0.12393
Fold 5 | Epoch  12 | Train RMSE=0.12342 | Valid RMSE=0.11537
Fold 5 | Epoch  13 | Train RMSE=0.12559 | Valid RMSE=0.14622
Fold 5 | Epoch  14 | Train RMSE=0.11895 | Valid RMSE=0.12179
Fold 5 | Epoch  15 | Train RMSE=0.12208 | Valid RMS